# Launch Batch Inference Job

## Batch Transformation Parameters
Update the parameters below to launch batch job

In [ ]:
# Path to the model.tar.gz output from the desired training job
model_data_s3_path = "s3://sagemaker-us-east-1-638418947120/segmentation-unet-1699635781-1a60/output/model.tar.gz"

# Instance Deployment Parameters, set the desired instance type and count
instance_type = "ml.m5.2xlarge"  # Ensure instance type has sufficient memory for model
instance_count = 1

# Output
output_bucket = "wwps-llnl-model-dev"
output_prefix= "batch_test/output/"

# Whether to wait for job. If true, blocks processing in this notebook and writes log output.
# If False, will return immediately and transform progress can be viewed in 
# SageMaker's Inference/Batch Transforms lists
wait_for_job = True

# Data to be processed. Uncomment and update the block that is appropriate
# For S3 Prefix processing
data = "s3://wwps-llnl-model-dev/batch_test/input/"
data_type = "S3Prefix"
content_type = "application/x-image" # streams the raw bytes to the model
max_payload_size = 25 # Maximum payload size in MB. Must be at least as large as a single image

# TODO: Add manifest options

## Create PyTorchModel object from existing training job

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorchModel
from sagemaker import get_execution_role

sagemaker_session = sagemaker.Session()
role = get_execution_role()

In [ ]:
# Create Environment Variables to pass to TorchServe
env_variables_dict = {
    # Update the maximum payload size for TS. Needs to match SageMaker's max payload size
    # Default for TS is 6,553,500 bytes
    # Convert max_payload_size in MB to Bytes
    "TS_DEFAULT_WORKERS_PER_MODEL": "4",
    "TS_MAX_REQUEST_SIZE": str(max_payload_size * 1024 * 1024)
}

# Create the PyTorchModel for the given model parameters
pytorch_model = PyTorchModel(
    model_data=model_data_s3_path,
    role=role,
    # Ensure framework version and py version match those used in LaunchTraining Jobs
    framework_version="2.0.1",
    py_version="py310",
    source_dir="../",
    entry_point="inference.py",
    model_server_workers=1,
    env=env_variables_dict,
)

In [ ]:
pytorch_model.serving_image_uri(region_name='us-east-1', instance_type=instance_type)

## Create Transform Object from Model
The transform object deploys the give model to the desired instance type and count.

**NOTE:** This step can take some time as this will create a new Model instance. It repackage the existing model.tar.gz file to include the `source_dir` given above.

In [ ]:
batch_transformer = pytorch_model.transformer(
    instance_type=instance_type, 
    instance_count=instance_count,
    output_path=f"s3://{output_bucket}/{output_prefix}",
    strategy="SingleRecord",
    max_payload=max_payload_size,
    max_concurrent_transforms=1,
    accept="application/x-image",
)

## Run Transform Job
We will point the transform job to an S3 Prefix and it will send all data in the prefix to the job. We can alternatively use a manifest file which lists the specific files to run rather than a prefix.

In [ ]:
batch_transformer.transform(
    data=data,
    data_type=data_type,
    content_type=content_type,
    wait=wait_for_job
)